# Stage 2: DPO Training on LoRA-tuned Mistral 7B

**Environment:** Kaggle GPU (P100/T4)  
**Estimated time:** ~1-2 hours  
**Input:** SFT adapter from Stage 1 + preference dataset  
**Output:** DPO-aligned LoRA adapter in `/kaggle/working/dpo_adapter/`

## Key fix vs. previous runs
The SFT adapter is **merged into the base model** before DPO begins.
A **fresh LoRA adapter** is created for DPO training, and `DPOTrainer`
uses the merged SFT model (with new adapter disabled) as the reference.
This ensures the KL penalty is computed against the SFT policy — not the
raw pretrained Mistral base, which was the root cause of mode collapse.

## 1. Install Dependencies

In [1]:
!pip install -q trl peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 13.9 MB/s eta 0:00:00


   ━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/60.7 MB 194.4 MB/s eta 0:00:01

   ━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/60.7 MB 231.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━ 30.2/60.7 MB 230.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━ 38.1/60.7 MB 230.2 MB/s eta 0:00:01

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 242.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 242.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 242.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 60.7/60.7 MB 242.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.1 MB/s eta 0:00:00


## 2. Verify Data Paths
Run this first to find exact paths on Kaggle.

In [2]:
import os
print("=== All files in /kaggle/input/ ===")
for root, dirs, files in os.walk('/kaggle/input/'):
    for f in files:
        print(os.path.join(root, f))

=== All files in /kaggle/input/ ===
/kaggle/input/datasets/aadityae/sft-adapter/adapter_model.safetensors
/kaggle/input/datasets/aadityae/sft-adapter/adapter_config.json
/kaggle/input/datasets/aadityae/sft-adapter/README.md
/kaggle/input/datasets/aadityae/sft-adapter/tokenizer.json
/kaggle/input/datasets/aadityae/sft-adapter/tokenizer_config.json
/kaggle/input/datasets/aadityae/sft-adapter/chat_template.jinja
/kaggle/input/datasets/aadityae/hr-preference-data/train.json
/kaggle/input/datasets/aadityae/hr-preference-data/test.json


## 3. Imports & Configuration
**IMPORTANT:** Update `SFT_ADAPTER_DIR` and `DATASET_PATH` below based on the output from the cell above.

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from trl import DPOTrainer

# ── Hyperparameters ──
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

# Quantization — float16 NOT bfloat16 (P100 doesn't support bf16)
LOAD_IN_4BIT = True
BNB_4BIT_QUANT_TYPE = "nf4"
BNB_4BIT_COMPUTE_DTYPE = torch.float16
BNB_4BIT_USE_DOUBLE_QUANT = True

# LoRA for DPO (fresh adapter — NOT reusing SFT adapter directly)
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# DPO-specific — conservative settings to avoid mode collapse
DPO_BETA = 0.3             # 0.1 was too low (divergence), 0.5 too conservative
DPO_EPOCHS = 1             # 1 epoch on small datasets to avoid overfitting
DPO_BATCH_SIZE = 4
DPO_GRADIENT_ACCUMULATION_STEPS = 4
DPO_LEARNING_RATE = 5e-6   # higher than before since we train a FRESH LoRA
DPO_MAX_LENGTH = 1024
DPO_MAX_PROMPT_LENGTH = 512
DPO_SAVE_STEPS = 500
DPO_LOGGING_STEPS = 10
DPO_WARMUP_STEPS = 50      # warmup_ratio is deprecated on Kaggle TRL

# ══════════════════════════════════════════════════
# UPDATE THESE PATHS based on cell 2 output above!
# ══════════════════════════════════════════════════
SFT_ADAPTER_DIR = "/kaggle/input/datasets/aadityae/sft-adapter"  # <-- update if different
DATASET_PATH = "/kaggle/input/datasets/aadityae/hr-preference-data/train.json"  # <-- update if different

DPO_OUTPUT_DIR = "./dpo_output"
DPO_ADAPTER_DIR = "/kaggle/working/dpo_adapter"

# Resume training (set to checkpoint path or None)
RESUME_FROM_CHECKPOINT = None

# Disable W&B to avoid team permission issues
os.environ["WANDB_DISABLED"] = "true"

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 4. Load Preference Dataset
DPO requires three columns: `prompt`, `chosen`, `rejected`.

In [4]:
raw_ds = load_dataset("json", data_files=DATASET_PATH, split="train")

# Keep only the columns DPO needs
keep_cols = ["prompt", "chosen", "rejected"]
remove_cols = [c for c in raw_ds.column_names if c not in keep_cols]
if remove_cols:
    raw_ds = raw_ds.remove_columns(remove_cols)

print(f"Loaded {len(raw_ds)} preference pairs")
print(f"Columns: {raw_ds.column_names}")
print(f"\nSample:")
print(f"  Prompt:   {raw_ds[0]['prompt'][:100]}...")
print(f"  Chosen:   {raw_ds[0]['chosen'][:100]}...")
print(f"  Rejected: {raw_ds[0]['rejected'][:100]}...")

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 648 preference pairs
Columns: ['prompt', 'chosen', 'rejected']

Sample:
  Prompt:   What expenses are reimbursable during business travel?...
  Chosen:   During business travel, the following expenses are reimbursable:

- **Meals**: 
  - Company-provided...
  Rejected: During business travel, the following expenses are reimbursable:

1. Meals while traveling on Compan...


## 5. Load Base Model + Merge SFT Adapter

**Critical:** We merge the SFT adapter weights into the base model, then
let `DPOTrainer` create a fresh LoRA adapter for DPO training.

Previous approach (loading SFT as a trainable PeftModel) caused mode collapse
because the reference model became the raw Mistral base instead of the SFT model.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_quant_type=BNB_4BIT_QUANT_TYPE,
    bnb_4bit_compute_dtype=BNB_4BIT_COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=BNB_4BIT_USE_DOUBLE_QUANT,
)

# Step 1: Load base model (quantized)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # left-padding is standard for DPO

# Step 2: Load the SFT adapter
sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_DIR, is_trainable=False)
print(f"SFT adapter loaded from: {SFT_ADAPTER_DIR}")

# Step 3: Merge SFT adapter into base model weights
# After merging, the model IS the SFT model (no adapter layer).
# DPOTrainer will then add a FRESH LoRA on top and use the merged
# weights (adapter-disabled = SFT model) as the reference policy.
model = sft_model.merge_and_unload()
print("SFT adapter merged into base model.")

# Step 4: Prepare for QLoRA training
model = prepare_model_for_kbit_training(model)

print(f"Base model: {BASE_MODEL}")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## 6. Configure DPO Training

Fresh LoRA config for DPO + training arguments with conservative hyperparameters.

In [ ]:
from trl import DPOConfig

# Fresh LoRA adapter for DPO (will be applied by DPOTrainer)
dpo_lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
training_args = DPOConfig(
    output_dir=DPO_OUTPUT_DIR,
    num_train_epochs=DPO_EPOCHS,
    per_device_train_batch_size=DPO_BATCH_SIZE,
    gradient_accumulation_steps=DPO_GRADIENT_ACCUMULATION_STEPS,
    learning_rate=DPO_LEARNING_RATE,
    warmup_steps=DPO_WARMUP_STEPS,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    fp16=False,   # P100 has issues with fp16 grad scaler + 4bit
    bf16=False,   # P100 doesn't support bf16
    gradient_checkpointing=True,
    logging_steps=DPO_LOGGING_STEPS,
    save_steps=DPO_SAVE_STEPS,
    save_total_limit=3,
    report_to="none",
    run_name="dpo-lora-mistral7b",
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    beta=DPO_BETA,
    max_length=DPO_MAX_LENGTH,
)

print(f"DPO beta: {DPO_BETA}")
print(f"Learning rate: {DPO_LEARNING_RATE}")
print(f"Effective batch size: {DPO_BATCH_SIZE * DPO_GRADIENT_ACCUMULATION_STEPS}")
print(f"Epochs: {DPO_EPOCHS}")
print(f"Max length: {DPO_MAX_LENGTH}")

## 7. Initialize DPO Trainer

`peft_config` tells DPOTrainer to create a fresh LoRA on the merged SFT model.
With no explicit `ref_model`, DPOTrainer uses the adapter-disabled version
(= merged SFT weights) as the reference policy. This is correct.

In [ ]:
import trl
print(f"TRL version: {trl.__version__}")

# Try new API first, fall back to old API
try:
    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,           # uses adapter-disabled model = merged SFT
        args=training_args,
        train_dataset=raw_ds,
        processing_class=tokenizer,
        peft_config=dpo_lora_config,  # fresh LoRA for DPO
    )
    print("DPOTrainer initialized (new API)")
except TypeError as e:
    print(f"New API failed: {e}")
    print("Trying old API...")
    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,
        args=training_args,
        train_dataset=raw_ds,
        tokenizer=tokenizer,
        peft_config=dpo_lora_config,
    )
    print("DPOTrainer initialized (old API)")

print(f"\nTraining samples: {len(raw_ds)}")
print(f"Checkpoints every {DPO_SAVE_STEPS} steps")

# Verify the model now has a fresh LoRA adapter
dpo_trainer.model.print_trainable_parameters()

## 8. Train

In [9]:
dpo_trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss


## 9. Save DPO Adapter

In [ ]:
os.makedirs(DPO_ADAPTER_DIR, exist_ok=True)

dpo_trainer.model.save_pretrained(DPO_ADAPTER_DIR)
tokenizer.save_pretrained(DPO_ADAPTER_DIR)

print(f"DPO adapter saved to {DPO_ADAPTER_DIR}")

for f in os.listdir(DPO_ADAPTER_DIR):
    size = os.path.getsize(os.path.join(DPO_ADAPTER_DIR, f))
    print(f"  {f}: {size / 1e6:.1f} MB")

## 10. Zip for Download

In [ ]:
!zip -r /kaggle/working/dpo_adapter.zip /kaggle/working/dpo_adapter/
print("\nZipped! Save Version -> Output tab to download.")

## 11. Quick Sanity Check
Compare DPO model output vs SFT-only output on the same prompt.

In [ ]:
# Switch to inference mode — gradient checkpointing breaks generation
dpo_trainer.model.gradient_checkpointing_disable()
dpo_trainer.model.config.use_cache = True
dpo_trainer.model.eval()

test_prompt = "[INST] What is the company's policy on remote work? [/INST]"
inputs = tokenizer(test_prompt, return_tensors="pt").to(dpo_trainer.model.device)

with torch.no_grad():
    outputs = dpo_trainer.model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        repetition_penalty=1.1,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("=== DPO Model ===")
print(response)

In [ ]:
# SFT-only model output (disable DPO adapter to get reference = merged SFT)
with dpo_trainer.model.disable_adapter():
    inputs = tokenizer(test_prompt, return_tensors="pt").to(dpo_trainer.model.device)
    with torch.no_grad():
        outputs = dpo_trainer.model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
        )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("=== SFT Model (reference) ===")
print(response)